In [1]:
import cv2
import torch
import numpy as np
from paddleocr import TextDetection
from itertools import groupby
import cv2
import onnxruntime as ort
sess_options = ort.SessionOptions()
sess_options.enable_profiling = True
itoc = torch.load("data/processed/vocab.pt")['itoc']

def process_single_sample(img):
    # BGR → Grayscale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Resize to (H=32, W=128)
    img = cv2.resize(img, (128, 32), interpolation=cv2.INTER_LINEAR)

    # Convert to float32 in [0, 1]
    img = img.astype(np.float32) / 255.0

    # Add channel and batch dimensions
    img = img[None, None, :, :]

    return img

def ctc_decoder(logits):
    labels = np.argmax(logits, axis=-1)
    decoded_texts = []
    decoded_indices = []
    for i in range(labels.shape[0]):
        ans = ""
        idx = []
        
        ## merge repeats
        merged_list = [k for k,_ in groupby(labels[i])]
        
        ## remove blanks
        for p in merged_list:
            if p != 0:
                ans += itoc[int(p)]
                idx.append(int(p))

        
        decoded_texts.append(ans)
        decoded_indices.append(idx)
        
    return decoded_texts, decoded_indices

2026-10-15 16:42:38.825662364 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
import torch
import json
itoc = torch.load("rec_data/processed/vocab.pt")['itoc']

with open('inference/itoc.json', 'w') as f:
    json.dump(itoc, f)

In [2]:
det_model = TextDetection(
            model_name="PP-OCRv6_tiny_det",
            engine="onnxruntime",
            device="cpu",
        )

Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/ritesh/.paddlex/official_models/PP-OCRv6_tiny_det_onnx`.


In [3]:
rec_model =  ort.InferenceSession(
    "crnn_fp32.onnx",
    providers=["CPUExecutionProvider"]
)

In [4]:
image_path = "tmp_images/27_State_1.png"
image = cv2.imread(image_path)

In [5]:
det_results = det_model.predict(
            input=image,
            batch_size=1,
        )

In [6]:

det_data = next(iter(det_results)).json["res"]

In [7]:
polygons = det_data["dt_polys"]

In [8]:
mask = np.zeros(image.shape[:2], dtype=np.uint8)

In [9]:
mask.shape

(63, 153)

In [10]:
text_results = []

In [11]:
for polygon in polygons:
    polygon = np.asarray(polygon, dtype=np.int32)

    # Create mask for inpainting
    cv2.fillPoly(mask, [polygon], 255)

    # Axis-aligned crop for first implementation
    x, y, w, h = cv2.boundingRect(polygon)

    left = max(0, x)
    top = max(0, y)
    right = min(image.shape[1], x + w)
    bottom = min(image.shape[0], y + h)

    crop = image[top:bottom, left:right]

    if crop.size == 0:
        continue

    crop_img = process_single_sample(crop)
    out = rec_model.run(
    ["logits"],
    {"images": crop_img})[0]
    pred_texts, pred_labels = ctc_decoder(out.transpose(1, 0, 2))
    text_results.append({
    "word": pred_texts,
    "bounding_box": [y, x, bottom, right],})

In [12]:
text_results

[{'word': ['State'], 'bounding_box': [0, 0, 63, 152]}]

In [13]:
import cv2

def visualize_ocr(image, results, display_width=1000):
    h, w = image.shape[:2]

    # Resize for visualization
    scale = display_width / w
    new_h = int(h * scale)

    image = cv2.resize(image, (display_width, new_h))

    for result in results:
        text = result["word"]
        if isinstance(text, list):
            text = text[0]

        top, left, bottom, right = result["bounding_box"]

        # Scale bounding-box coordinates
        top = int(top * scale)
        left = int(left * scale)
        bottom = int(bottom * scale)
        right = int(right * scale)

        cv2.rectangle(
            image, (left, top), (right, bottom),
            (0, 255, 0), 2
        )

        # Draw text with a fixed, readable font size
        cv2.putText(
            image,
            str(text),
            (left, max(20, top - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            1,
            cv2.LINE_AA
        )

    return image

In [14]:
print(type(text_results))
print(text_results[0])
print(type(text_results[0]["word"]))

<class 'list'>
{'word': ['State'], 'bounding_box': [0, 0, 63, 152]}
<class 'list'>


In [15]:
image = cv2.imread(image_path)

visualized = visualize_ocr(image, text_results)

cv2.imwrite("sample_annotated.png", visualized)

True